In [3]:
import json
import pandas as pd
import numpy as np

# Load your local or Drive JSON file
file_path = '/content/cardium_clinical_data_woe_wnm_standarized_f_normalized.json'

with open(file_path, 'r') as f:
    data = json.load(f)

# Convert to Pandas DataFrame
df = pd.DataFrame(data)

# 1. Total records count
total_samples = len(df)

# 2. Target Class Distribution
chd_counts = df['CHD'].value_counts()
chd_percentages = df['CHD'].value_counts(normalize=True) * 100

print("="*50)
print(f"     CARDIUM DATASET: STATISTICAL ANALYSIS")
print("="*50)
print(f"Total Patients / Records: {total_samples}")
print("\n--- CHD Target Class Breakdown ---")
for cls, count in chd_counts.items():
    pct = chd_percentages[cls]
    label = "Pathological (CHD Positive)" if cls == 1 else "Normal (CHD Negative)"
    print(f"Class {cls} [{label}]: {count} samples ({pct:.2f}%)")

# 3. Missing Value Analysis (-1 flags)
print("\n--- Missing / Padded Values (-1) Summary ---")
for col in ['platelets', 'hemoglobin', 'white_blood_cells', 'neutrophils']:
    if col in df.columns:
        # Extract first value if array, or check direct value
        vals = df[col].apply(lambda x: x[0] if isinstance(x, list) else x)
        missing_count = (vals == -1).sum()
        print(f"Feature '{col}': {missing_count} missing entries ({(missing_count/total_samples)*100:.1f}%)")

     CARDIUM DATASET: STATISTICAL ANALYSIS
Total Patients / Records: 1104

--- CHD Target Class Breakdown ---
Class 0 [Normal (CHD Negative)]: 1030 samples (93.30%)
Class 1 [Pathological (CHD Positive)]: 74 samples (6.70%)

--- Missing / Padded Values (-1) Summary ---
Feature 'platelets': 225 missing entries (20.4%)
Feature 'hemoglobin': 220 missing entries (19.9%)
Feature 'white_blood_cells': 277 missing entries (25.1%)
Feature 'neutrophils': 370 missing entries (33.5%)


In [4]:
import os
import glob
import json
import pandas as pd

# Path to the specific Drive folder
folder_id = "/content/drive/MyDrive/fetal/CARDIUM_dataset"

# Search for folder across Drive mount
drive_root = "/content/drive/MyDrive"
target_folder = None

for root, dirs, files in os.walk(drive_root):
    if folder_id in root or "fetal" in dirs:
        target_folder = os.path.join(root, "fetal") if "fetal" in dirs else root
        break

if not target_folder:
    # Default path if folder name is 'fetal' in MyDrive
    target_folder = "/content/drive/MyDrive/fetal"

print("="*60)
print(f"📁 ANALYZING FOLDER: {target_folder}")
print("="*60)

if os.path.exists(target_folder):
    all_files = os.listdir(target_folder)
    print(f"Total files/items in folder: {len(all_files)}\n")

    json_files = [f for f in all_files if f.endswith('.json')]
    csv_files = [f for f in all_files if f.endswith('.csv')]

    print("Files found:")
    for f in all_files:
        print(f"  - {f}")

    # Analyze JSON files for record count & target classes (CHD)
    for jf in json_files:
        file_path = os.path.join(target_folder, jf)
        print("\n" + "-"*50)
        print(f"📊 Analyzing File: {jf}")
        print("-"*50)

        try:
            with open(file_path, 'r') as f:
                data = json.load(f)

            df = pd.DataFrame(data)
            print(f"Total Patient Records: {len(df)}")

            # Check for CHD target column
            if 'CHD' in df.columns:
                counts = df['CHD'].value_counts()
                percentages = df['CHD'].value_counts(normalize=True) * 100
                print("\nTarget Class Distribution ('CHD'):")
                for cls_val, cnt in counts.items():
                    label = "Pathological / CHD Positive" if cls_val == 1 else "Normal / CHD Negative"
                    print(f"  Class {cls_val} [{label}]: {cnt} records ({percentages[cls_val]:.2f}%)")
            else:
                print("Available Columns:", list(df.columns[:10]), "...")
        except Exception as e:
            print(f"Could not parse JSON content: {e}")

    # Analyze CSV files if present
    for cf in csv_files:
        file_path = os.path.join(target_folder, cf)
        print("\n" + "-"*50)
        print(f"📊 Analyzing CSV File: {cf}")
        print("-"*50)
        df = pd.read_csv(file_path)
        print(f"Total Records: {len(df)}")
        if 'CHD' in df.columns or 'fetal_health' in df.columns:
            target_col = 'CHD' if 'CHD' in df.columns else 'fetal_health'
            print(f"\nTarget Class Distribution ('{target_col}'):")
            print(df[target_col].value_counts())

else:
    print(f"⚠️ Target folder not found at {target_folder}. Please check path in Mounted Drive.")

📁 ANALYZING FOLDER: /content/drive/MyDrive/fetal
Total files/items in folder: 2

Files found:
  - cardium_images
  - CARDIUM_dataset


In [5]:
import os
import glob
import shutil
import random
from sklearn.model_selection import train_test_split

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
# Source folder containing your raw images
SOURCE_IMAGE_DIR = "/content/drive/MyDrive/fetal/cardium_images"  # Update if pointing to 'fetal' folder
OUTPUT_BALANCED_DIR = "./dataset_balanced_1to1"

SEED = 42
random.seed(SEED)

# ==========================================
# 2. GATHER ALL NORMAL & CHD IMAGES
# ==========================================
print("[*] Searching for ultrasound images...")

# Gather Normal Images
normal_paths = [
    os.path.join(SOURCE_IMAGE_DIR, "normal"),
    os.path.join(SOURCE_IMAGE_DIR, "Normal"),
    "./dataset_split/train/Normal",
    "./dataset_split/val/Normal",
    "./dataset_split/test/Normal"
]

all_normal_files = []
for path in normal_paths:
    if os.path.exists(path):
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG']:
            all_normal_files.extend(glob.glob(os.path.join(path, ext)))

all_normal_files = sorted(list(set(all_normal_files)))

# Gather CHD Images
chd_subfolders = ["train_chd", "val_chd", "test_chd", "fchd", "chd", "CHD"]
all_chd_files = []
for subfolder in chd_subfolders:
    path = os.path.join(SOURCE_IMAGE_DIR, subfolder)
    if os.path.exists(path):
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG']:
            all_chd_files.extend(glob.glob(os.path.join(path, ext)))

# Also check inside existing split folders
for split in ['train', 'val', 'test']:
    path = f"./dataset_split/{split}/CHD"
    if os.path.exists(path):
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG']:
            all_chd_files.extend(glob.glob(os.path.join(path, ext)))

all_chd_files = sorted(list(set(all_chd_files)))

print(f" -> Found {len(all_normal_files)} Normal images.")
print(f" -> Found {len(all_chd_files)} CHD images.")

if len(all_normal_files) == 0 or len(all_chd_files) == 0:
    raise FileNotFoundError("Could not find both Normal and CHD image files. Please verify your source image directory.")

# ==========================================
# 3. SAMPLE EQUAL IMAGES (STRICT 1:1 RATIO)
# ==========================================
# Set size based on minority class to keep 1:1 balance
min_count = min(len(all_normal_files), len(all_chd_files))

sampled_normals = random.sample(all_normal_files, min_count)
sampled_chd = random.sample(all_chd_files, min_count)

print(f"\n[+] Extracted {min_count} Normal and {min_count} CHD images for 1:1 balance.")

# ==========================================
# 4. STRATIFIED TRAIN / VAL / TEST SPLIT
# ==========================================
def split_images(file_list):
    # 75% Train, 25% Temp
    train_f, temp_f = train_test_split(file_list, test_size=0.25, random_state=SEED, shuffle=True)
    # 15% Val, 10% Test
    val_f, test_f = train_test_split(temp_f, test_size=0.40, random_state=SEED, shuffle=True)
    return train_f, val_f, test_f

norm_train, norm_val, norm_test = split_images(sampled_normals)
chd_train, chd_val, chd_test = split_images(sampled_chd)

# ==========================================
# 5. SAVE INTO STRUCTURED DIRECTORIES
# ==========================================
splits_map = {
    'train': {'Normal': norm_train, 'CHD': chd_train},
    'val':   {'Normal': norm_val,   'CHD': chd_val},
    'test':  {'Normal': norm_test,  'CHD': chd_test}
}

for split_name, categories in splits_map.items():
    for cat_name, file_list in categories.items():
        target_dir = os.path.join(OUTPUT_BALANCED_DIR, split_name, cat_name)
        os.makedirs(target_dir, exist_ok=True)

        for idx, src_path in enumerate(file_list):
            ext = os.path.splitext(src_path)[1]
            dest_filename = f"{cat_name.lower()}_{split_name}_{idx:03d}{ext}"
            shutil.copy(src_path, os.path.join(target_dir, dest_filename))

print("\n" + "="*50)
print("     BALANCED IMAGE DATASET SUMMARY")
print("="*50)
print(f"Train Set : {len(norm_train)} Normal | {len(chd_train)} CHD (Total: {len(norm_train) + len(chd_train)})")
print(f"Val Set   : {len(norm_val)} Normal | {len(chd_val)} CHD (Total: {len(norm_val) + len(chd_val)})")
print(f"Test Set  : {len(norm_test)} Normal | {len(chd_test)} CHD (Total: {len(norm_test) + len(chd_test)})")
print("="*50)
print(f"[+] Clean 1:1 image dataset ready at: '{OUTPUT_BALANCED_DIR}'")

[*] Searching for ultrasound images...
 -> Found 223 Normal images.
 -> Found 0 CHD images.


FileNotFoundError: Could not find both Normal and CHD image files. Please verify your source image directory.